# 03 — ConvNeXt-Tiny E5 Integrated — APTOS2019

**Studi 3: Reinvent (eksploratori).** Full fine-tuning ConvNeXt-Tiny dengan categorical CE, auxiliary ordinal objective (0.25), dan deferred inverse-square-root class weighting mulai epoch 11. Seed 42, 20 epoch, BF16, batch efektif 32.

**Hasil faktual:** best checkpoint epoch 16 dipilih dengan validation macro-F1; frozen final test dilaksanakan satu kali terhadap **366** gambar. Notebook ini hanya membaca laporan, tidak menjalankan test ulang. [Kode model](../novelty_integrated/e5/model.py) · [Runner](../novelty_integrated/e5/runner.py).

In [ ]:
from pathlib import Path
from IPython.display import display, Image, SVG, Markdown
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'README.md').exists() and (p/'reproduction').exists()), None)
assert ROOT is not None, 'Jalankan notebook dari dalam checkout GitHub proyek.'
print('Project:', ROOT)

## Hasil terkunci — dibaca dari JSON dan CSV

Validation macro-F1 **0.705791**; test macro-F1 **0.640456**; test accuracy **82.51%**; QWK **0.891194**. Angka ini bukan hasil CAD-ViT atau B0–B4.

In [ ]:
import json,csv
P=ROOT/'results/APTOS2019/e5_integrated'
s=json.loads((P/'summary.json').read_text())
t=json.loads((P/'test_summary.json').read_text())
with (P/'history.csv').open() as f: h=list(csv.DictReader(f))
assert s['status']=='VALIDATION_COMPLETE' and t['status']=='FINAL_TEST_COMPLETE'
assert s['best']['epoch']==t['checkpoint_epoch']==16
assert len(h)==20 and t['test_samples']==366
print('Validation F1:',s['best']['validation']['macro_f1'])
print('Test F1:',t['metrics']['macro_f1'])
print('Test QWK:',t['metrics']['quadratic_weighted_kappa'])

## Learning curves (20 epoch)

![Training and validation F1](../results/APTOS2019/e5_integrated/learning_curves.svg)

## Confusion matrix — validation

Baris = aktual, kolom = prediksi.

![Validation matrix](../results/APTOS2019/e5_integrated/validation_confusion_matrix.svg)

## Confusion matrix — frozen final test

Baris = aktual, kolom = prediksi.

![Test matrix](../results/APTOS2019/e5_integrated/test_confusion_matrix.svg)

In [ ]:
for x in t['metrics']['per_grade']:
    print(f"Grade {x['grade']}: precision={x['precision']:.4f} recall={x['recall']:.4f} F1={x['f1']:.4f} correct={x['correct']}/{x['support']}")

## Pembahasan kritis

Grade 3 hanya **4/19 benar**, test F1 **0.2500**. Ada **15** tambahan kemunculan citra identik dalam frozen test, tanpa SHA/group overlap dengan train/validation pada audit. Semua **366** baris dievaluasi tanpa penghapusan. Penurunan validation→test macro-F1 menjadi keterbatasan generalisasi, bukan bukti bahwa ordinal learning/deferred weights berhasil mengungguli B1. Belum ada perbandingan final test yang terkontrol.

**Status DDR E5:** belum dibuktikan final dalam artefak ini; tidak ada angka DDR yang direkayasa.